# Explore the COAWST US East Coast and Gulf of Mexico Forecast Archive Dataset

This notebook opens the COAWST dataset from the **Icechunk** virtual store on S3
(`s3://usgs-coawst/useast-archive/icechunk/coawst-useast.icechunk`) and explores it
with Dask for parallel computation.

In [1]:
import xarray as xr
import hvplot.xarray
import cf_xarray
import numpy as np
import panel as pn
from matplotlib import path
import xoak
import icechunk
from icechunk import (
    ManifestConfig,
    ManifestSplitCondition,
    ManifestSplitDimCondition,
    ManifestSplittingConfig,
)

## Open Dataset

Open the Icechunk store anonymously (the data is on the AWS Open Data Program,
`s3://usgs-coawst`), then open the dataset with xarray backed by Dask arrays.

In [2]:
bucket = 'usgs-coawst'
region = 'us-west-2'
prefix = 'useast-archive/icechunk/coawst-useast.icechunk'
TIME_DIM = 'ocean_time'

# Manifest splitting mirrors the config used when the store was built.
# Icechunk uses it for lazy manifest loading: only the manifest files
# covering the requested time range are fetched, not the entire store.
split_config = ManifestSplittingConfig.from_dict({
    ManifestSplitCondition.AnyArray(): {
        ManifestSplitDimCondition.DimensionName(TIME_DIM): 365 * 24
    }
})
config = icechunk.RepositoryConfig(manifest=ManifestConfig(splitting=split_config))
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f's3://{bucket}/',
        store=icechunk.s3_store(region=region, anonymous=True),
    )
)
creds = icechunk.containers_credentials(
    {f's3://{bucket}/': icechunk.s3_credentials(anonymous=True)}
)
storage = icechunk.s3_storage(bucket=bucket, prefix=prefix, region=region, anonymous=True)
repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=creds)
session = repo.readonly_session('main')

In [3]:
%%time
ds = xr.open_zarr(session.store, consolidated=False)
ds

/home/rsignell/miniforge3/envs/coawst-icechunk/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


CPU times: user 648 ms, sys: 329 ms, total: 976 ms
Wall time: 2.07 s


<xarray.Dataset> Size: 47TB
Dimensions:                 (ocean_time: 112392, s_w: 17, eta_rho: 336,
                             xi_rho: 896, tracer: 8, s_rho: 16, NST: 6,
                             boundary: 4, Nbed: 1, eta_v: 335, xi_v: 896,
                             eta_u: 336, xi_u: 895, eta_psi: 335, xi_psi: 895)
Coordinates:
  * ocean_time              (ocean_time) datetime64[ns] 899kB 2009-08-21T01:0...
  * s_w                     (s_w) float64 136B -1.0 -0.9375 ... -0.0625 0.0
    lat_rho                 (eta_rho, xi_rho) float64 2MB dask.array<chunksize=(168, 224), meta=np.ndarray>
    lon_rho                 (eta_rho, xi_rho) float64 2MB dask.array<chunksize=(168, 224), meta=np.ndarray>
  * s_rho                   (s_rho) float64 128B -0.9688 -0.9062 ... -0.03125
    lon_v                   (eta_v, xi_v) float64 2MB dask.array<chunksize=(168, 224), meta=np.ndarray>
    lat_v                   (eta_v, xi_v) float64 2MB dask.array<chunksize=(168, 224), meta=np.ndarray>
    lat_u                   (eta_u, xi_u) float64 2MB dask.array<chunksize=(168, 224), meta=np.ndarray>
    lon_u                   (eta_u, xi_u) float64 2MB dask.array<chunksize=(168, 224), meta=np.ndarray>
    lat_psi                 (eta_psi, xi_psi) float64 2MB dask.array<chunksize=(168, 224), meta=np.ndarray>
    lon_psi                 (eta_psi, xi_psi) float64 2MB dask.array<chunksize=(168, 224), meta=np.ndarray>
Dimensions without coordinates: eta_rho, xi_rho, tracer, NST, boundary, Nbed,
                                eta_v, xi_v, eta_u, xi_u, eta_psi, xi_psi
Data variables: (12/187)
    AKs                     (ocean_time, s_w, eta_rho, xi_rho) float32 2TB dask.array<chunksize=(168, 1, 168, 224), meta=np.ndarray>
    Akp_bak                 float64 8B ...
    Akv_bak                 float64 8B ...
    Charnok_alpha           float64 8B ...
    Akt_bak                 (tracer) float64 64B dask.array<chunksize=(8,), meta=np.ndarray>
    Cs_r                    (s_rho) float64 128B dask.array<chunksize=(1,), meta=np.ndarray>
    ...                      ...
    vbar                    (ocean_time, eta_v, xi_v) float32 135GB dask.array<chunksize=(168, 168, 224), meta=np.ndarray>
    wetdry_mask_u           (ocean_time, eta_u, xi_u) float32 135GB dask.array<chunksize=(168, 168, 224), meta=np.ndarray>
    wetdry_mask_v           (ocean_time, eta_v, xi_v) float32 135GB dask.array<chunksize=(168, 168, 224), meta=np.ndarray>
    xl                      float64 8B ...
    zeta                    (ocean_time, eta_rho, xi_rho) float32 135GB dask.array<chunksize=(168, 168, 224), meta=np.ndarray>
    wetdry_mask_rho         (ocean_time, eta_rho, xi_rho) float32 135GB dask.array<chunksize=(168, 168, 224), meta=np.ndarray>
Attributes: (12/37)
    CPP_options:       COAWST, ANA_BPFLUX, ANA_BSFLUX, ANA_BTFLUX, ANA_FSOBC,...
    Conventions:       CF-1.4, SGRID-0.3
    NLM_LBC:           \nEDGE:         WEST   SOUTH  EAST   NORTH  \nzeta:   ...
    ana_file:          ROMS/Functionals/ana_btflux.h, ROMS/Functionals/ana_fs...
    bry_file_01:       ./forcings3/USE_coawst_bdy.nc
    clm_file_01:       ./forcings3/USE_coawst_clm.nc
    ...                ...
    svn_url:            
    tide_file:         ../forcings/tide_forc_USeast_grd16_osu_rev2.nc
    tiling:            006x004
    title:             COAWST ROMS SWAN
    type:              ROMS/TOMS history file
    var_info:          ROMS/External/varinfo.dat

The dataset has 14 years of hourly output across the US East and Gulf coasts.
Metadata and coordinates are loaded immediately; data variables remain lazy (Dask arrays).

Let's look at that metadata.  We can explore the different attributes and variables by clicking on the variables and icons below. 

In [4]:
ds.nbytes/1e12

46.941544690368

We can also explore a specific variable of interest:

In [5]:
var = 'Hwave'
da = ds[var]
da

<xarray.DataArray 'Hwave' (ocean_time: 112392, eta_rho: 336, xi_rho: 896)> Size: 135GB
dask.array<open_dataset-Hwave, shape=(112392, 336, 896), dtype=float32, chunksize=(168, 168, 224), chunktype=numpy.ndarray>
Coordinates:
  * ocean_time  (ocean_time) datetime64[ns] 899kB 2009-08-21T01:00:00 ... 202...
    lat_rho     (eta_rho, xi_rho) float64 2MB dask.array<chunksize=(168, 224), meta=np.ndarray>
    lon_rho     (eta_rho, xi_rho) float64 2MB dask.array<chunksize=(168, 224), meta=np.ndarray>
Dimensions without coordinates: eta_rho, xi_rho
Attributes:
    field:      Hwave, scalar, series
    grid:       grid
    location:   face
    long_name:  wind-induced significant wave height
    time:       ocean_time
    units:      meter

Use the CF conventions to identify the coordinate variables for longitude, latitude and time

In [6]:
x = da.cf['longitude']
y = da.cf['latitude']
t = da.cf['time']
print(x.name, y.name, t.name)

lon_rho lat_rho ocean_time


## Example: Load the entire spatial domain for a variable at a specific time step
Loading the entire spatial domain at a time step only requires reading 8 chunks of data, so it loads in a few seconds.  A dask cluster doesn't help much in this case as it's already fast.   


In [7]:
%%time
da2d = da.cf.sel(T='2012-10-29 12:00', method='nearest').load()

CPU times: user 1.02 s, sys: 1.02 s, total: 2.04 s
Wall time: 12.1 s


In [8]:
da2d.hvplot.quadmesh(x=x.name, y=y.name, rasterize=True, geo=True, tiles='OSM', cmap='viridis')

:DynamicMap   []
   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [lon_rho,lat_rho]   (wind-induced significant wave height)

## Example: Load a time series for a variable at a specific lon,lat location for a specified time range. 

To identify a point, we will start with its lat/lon coordinates.  If lon and lat were 1D coordinates, we could use lon,lat values to select using xarray, but instead we need to extract using indices, which we need to find.   For this we use the `xoak` package:

In [9]:
lat,lon = 42.5, -70.0  # Gulf of Maine, 100km east of Boston, MA

In [10]:
da.xoak.set_index([y.name, x.name], 'scipy_kdtree')

/tmp/ipykernel_668560/3835131389.py:1: FutureWarning: Setting the index via the xoak accessor is deprecated and will be removed in a future version. Instead of `.xoak.set_index(['lat_rho', 'lon_rho'], ...)`, use the Xarray API `.set_xindex(['lat_rho', 'lon_rho'], xarray.indexes.NDPointIndex, tree_adapter_cls=...)`
  da.xoak.set_index([y.name, x.name], 'scipy_kdtree')


In [11]:
ds_point = xr.Dataset({"lon": ("point", [lon]), "lat": ("point", [lat])})

Before we read the data, let's see how many chunks we will be reading:

In [12]:
da.xoak.sel(lat_rho=ds_point.lat, lon_rho=ds_point.lon).cf.sel(T='2012-10')

/tmp/ipykernel_668560/2027962924.py:1: FutureWarning: Data selection via `.xoak.sel()` is deprecated and will be removed in a future version. Instead of `.xoak.sel(...)`, use directly the Xarray API `.sel(...)` after setting an `xarray.indexes.NDPointIndex` with one of the tree adapter classes avaiable in Xoak.
  da.xoak.sel(lat_rho=ds_point.lat, lon_rho=ds_point.lon).cf.sel(T='2012-10')


<xarray.DataArray 'Hwave' (ocean_time: 744, point: 1)> Size: 3kB
dask.array<getitem, shape=(744, 1), dtype=float32, chunksize=(168, 1), chunktype=numpy.ndarray>
Coordinates:
  * ocean_time  (ocean_time) datetime64[ns] 6kB 2012-10-01 ... 2012-10-31T23:...
    lat_rho     (point) float64 8B 42.5
    lon_rho     (point) float64 8B -70.02
Dimensions without coordinates: point
Attributes:
    field:      Hwave, scalar, series
    grid:       grid
    location:   face
    long_name:  wind-induced significant wave height
    time:       ocean_time
    units:      meter

To load this one month means reading 5 chunks of data, so still don't need a cluster:

In [13]:
 da.xoak.sel(lat_rho=ds_point.lat, lon_rho=ds_point.lon).cf.sel(T='2012-10')

/tmp/ipykernel_668560/2473571537.py:1: FutureWarning: Data selection via `.xoak.sel()` is deprecated and will be removed in a future version. Instead of `.xoak.sel(...)`, use directly the Xarray API `.sel(...)` after setting an `xarray.indexes.NDPointIndex` with one of the tree adapter classes avaiable in Xoak.
  da.xoak.sel(lat_rho=ds_point.lat, lon_rho=ds_point.lon).cf.sel(T='2012-10')


<xarray.DataArray 'Hwave' (ocean_time: 744, point: 1)> Size: 3kB
dask.array<getitem, shape=(744, 1), dtype=float32, chunksize=(168, 1), chunktype=numpy.ndarray>
Coordinates:
  * ocean_time  (ocean_time) datetime64[ns] 6kB 2012-10-01 ... 2012-10-31T23:...
    lat_rho     (point) float64 8B 42.5
    lon_rho     (point) float64 8B -70.02
Dimensions without coordinates: point
Attributes:
    field:      Hwave, scalar, series
    grid:       grid
    location:   face
    long_name:  wind-induced significant wave height
    time:       ocean_time
    units:      meter

In [14]:
%%time
da1d = da.xoak.sel(lat_rho=ds_point.lat, lon_rho=ds_point.lon).cf.sel(T='2012-10').load()

<timed exec>:1: FutureWarning: Data selection via `.xoak.sel()` is deprecated and will be removed in a future version. Instead of `.xoak.sel(...)`, use directly the Xarray API `.sel(...)` after setting an `xarray.indexes.NDPointIndex` with one of the tree adapter classes avaiable in Xoak.


CPU times: user 783 ms, sys: 357 ms, total: 1.14 s
Wall time: 10.5 s


In [15]:
da1d.hvplot(x=t.name, grid=True)

:Curve   [ocean_time]   (wind-induced significant wave height)

How many chunks of data will we read to load the entire time series of record at a point?

In [16]:
da.xoak.sel(lat_rho=ds_point.lat, lon_rho=ds_point.lon)

/tmp/ipykernel_668560/4036444056.py:1: FutureWarning: Data selection via `.xoak.sel()` is deprecated and will be removed in a future version. Instead of `.xoak.sel(...)`, use directly the Xarray API `.sel(...)` after setting an `xarray.indexes.NDPointIndex` with one of the tree adapter classes avaiable in Xoak.
  da.xoak.sel(lat_rho=ds_point.lat, lon_rho=ds_point.lon)


<xarray.DataArray 'Hwave' (ocean_time: 112392, point: 1)> Size: 450kB
dask.array<transpose, shape=(112392, 1), dtype=float32, chunksize=(168, 1), chunktype=numpy.ndarray>
Coordinates:
  * ocean_time  (ocean_time) datetime64[ns] 899kB 2009-08-21T01:00:00 ... 202...
    lat_rho     (point) float64 8B 42.5
    lon_rho     (point) float64 8B -70.02
Dimensions without coordinates: point
Attributes:
    field:      Hwave, scalar, series
    grid:       grid
    location:   face
    long_name:  wind-induced significant wave height
    time:       ocean_time
    units:      meter

Since we now need to read 669 chunks of data, we should use a Dask cluster if we have access to one

### Parallelize with Dask 
We opened the dataset so that we can take advantage of parallel compute environments
using `dask`. We're going to start a cluster now so that future steps can take advantage
of this ability. 

This is an optional step, but speeds up data loading and processing significantly, especially 
when accessing data from the cloud.

There are many ways to [deploy a Dask cluster](https://docs.dask.org/en/stable/deploying.html#deploy-dask-clusters).   
Below each cell uses a different approach.   Use one of the approaches below or choose another method. 

In [17]:
#cluster_type = 'Coiled'    
#cluster_type = 'Coiled'
# cluster_type = 'Gateway'
cluster_type = 'Coiled'

#### Use LocalCluster
LocalCluster is available in any computing environment.  It uses the number of CPUs of the computer running the notebook to create a cluster. 

In [18]:
if cluster_type == 'Local':
    from dask.distributed import LocalCluster, Client
    cluster = LocalCluster()
    client = Client(cluster)

#### Use Coiled
[Coiled](https://www.coiled.io/) provides access to remote Dask clusters that can be used from anywhere.  It requires a Coiled account. 

In [19]:
if cluster_type == 'Coiled':
    import coiled
    cluster = coiled.Cluster(
        region="us-west-2",
        arm=True,   # run on ARM to save energy & cost
        worker_vm_types=["t4g.small"],  # cheap, small ARM instances, 2cpus, 2GB RAM
        worker_options={'nthreads':2},
        n_workers=30,
        wait_for_workers=False,
        compute_purchase_option="spot_with_fallback",
        name='coawst',   # Dask cluster name
        software='coawst-icechunk-arm',  # Conda environment name
        workspace='esip-lab',
        timeout=180   # leave cluster running for 3 min in case we want to use it again
    )

    client = cluster.get_client()

#### Use a Dask Gateway Cluster
[Dask Gateway](https://gateway.dask.org/) is a common way to spin up a Dask Cluster.  [Nebari](https://nebari.dev) and [DaskHub](https://github.com/dask/helm-chart) are popular ways of deploying a JupyterHub with Dask Gateway.  You can use a JupyterHub with DaskGateway for free by [signing up for access to the Microsoft Planetary Computer hub](https://planetarycomputer.microsoft.com/account/request).

In [20]:
%%time
if cluster_type == 'Gateway':
    from dask_gateway import Gateway

    gateway = Gateway()  # instantiate Dask gateway 

    # Cluster options on Nebari 
    options = gateway.cluster_options()
    options.conda_environment='global/global-pangeo'  # comment out for Daskhub or Planetary Computer
    options.profile = 'Small Worker'   # comment out for Daskhub or Planetary Computer

    # Create a Dask Gateway cluster
    cluster = gateway.new_cluster(options)

    # Get the Dask client for the Dask Gateway cluster
    client = cluster.get_client()

    # Scale the cluster
    cluster.adapt(minimum=4, maximum=30)

CPU times: user 3 μs, sys: 1e+03 ns, total: 4 μs
Wall time: 5.25 μs


In [21]:
%%time
if cluster_type == 'Nebari':
    import sys, os
    sys.path.append(os.path.join(os.environ['HOME'],'shared','users','lib'))
    import nebari_tools as nbt

    aws_profile = 'esip-qhub'
    aws_region = 'us-west-2'
    endpoint_url = f's3.{aws_region}.amazonaws.com'

    nbt.set_credentials(profile=aws_profile, region=aws_region, endpoint_url=endpoint_url)
    worker_max = 30

    client, cluster = nbt.start_dask_cluster(profile=aws_profile, worker_max=worker_max, 
                                          region=aws_region, use_existing_cluster=True,
                                          adaptive_scaling=True, wait_for_cluster=True, 
                                          worker_profile='Small Worker', 
                                          propagate_env=True)

CPU times: user 2 μs, sys: 1 μs, total: 3 μs
Wall time: 5.01 μs


In [22]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 12,Total memory: 6.64 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34747,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:41589,Total threads: 3
Dashboard: http://127.0.0.1:44325/status,Memory: 1.66 GiB
Nanny: tcp://127.0.0.1:41505,


Load the entire time series:

In [23]:
%%time
ds_selection = da.xoak.sel(lat_rho=ds_point.lat, lon_rho=ds_point.lon).load()       

<timed exec>:1: FutureWarning: Data selection via `.xoak.sel()` is deprecated and will be removed in a future version. Instead of `.xoak.sel(...)`, use directly the Xarray API `.sel(...)` after setting an `xarray.indexes.NDPointIndex` with one of the tree adapter classes avaiable in Xoak.


CPU times: user 16.3 s, sys: 3.81 s, total: 20.1 s
Wall time: 5min 6s


In [24]:
ds_selection.hvplot(x=t.name, grid=True) 

:Curve   [ocean_time]   (wind-induced significant wave height)

## Example: Compute the time mean for a variable over the entire domain for a specific time period

In [25]:
%%time
da_mean = da.cf.sel(T=slice('2016-01-01 00:00','2017-01-01 00:00')).mean(dim=t.name).compute()

CPU times: user 12.7 s, sys: 2.68 s, total: 15.4 s
Wall time: 2min 30s


In [26]:
da_mean.hvplot.quadmesh(x=x.name, y=y.name, rasterize=True, geo=True, tiles='OSM', cmap='viridis')

:DynamicMap   []
   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [lon_rho,lat_rho]   (wind-induced significant wave height)

## Example: Subset a time and space region and export to NetCDF

In [27]:
def bbox2ij(lon,lat,bbox=[-160., -155., 18., 23.]):
    """Return indices for i,j that will completely cover the specified bounding box.     
    i0,i1,j0,j1 = bbox2ij(lon,lat,bbox)
    lon,lat = 2D arrays that are the target of the subset
    bbox = list containing the bounding box: [lon_min, lon_max, lat_min, lat_max]

    Example
    -------  
    >>> i0,i1,j0,j1 = bbox2ij(lon_rho,lat_rho,[-71, -63., 39., 46])
    >>> h_subset = nc.variables['h'][j0:j1,i0:i1]       
    """
    bbox=np.array(bbox)
    mypath=np.array([bbox[[0,1,1,0]],bbox[[2,2,3,3]]]).T
    p = path.Path(mypath)
    points = np.vstack((lon.ravel(),lat.ravel())).T   
    n,m = np.shape(lon)
    inside = p.contains_points(points).reshape((n,m))
    ii,jj = np.meshgrid(range(m),range(n))
    return min(ii[inside]),max(ii[inside]),min(jj[inside]),max(jj[inside])

In [28]:
bbox = [-76.63290610753754, -73.55671530588432, 37.57888442021855, 41.225532965406224]   # DRB

In [29]:
i0,i1,j0,j1 = bbox2ij(x.values, y.values, bbox=bbox)
print(i0,i1,j0,j1)

518 603 253 328


In [30]:
ds_drb = ds[['temp', 'salt', 'Hwave']].isel(eta_rho=slice(j0,j1), xi_rho=slice(i0,i1))

In [31]:
ds_drb

<xarray.Dataset> Size: 95GB
Dimensions:     (ocean_time: 112392, s_rho: 16, eta_rho: 75, xi_rho: 85)
Coordinates:
  * ocean_time  (ocean_time) datetime64[ns] 899kB 2009-08-21T01:00:00 ... 202...
  * s_rho       (s_rho) float64 128B -0.9688 -0.9062 ... -0.09375 -0.03125
    lat_rho     (eta_rho, xi_rho) float64 51kB 36.5 36.52 36.54 ... 42.2 42.22
    lon_rho     (eta_rho, xi_rho) float64 51kB -76.16 -76.11 ... -74.11 -74.06
Dimensions without coordinates: eta_rho, xi_rho
Data variables:
    temp        (ocean_time, s_rho, eta_rho, xi_rho) float32 46GB dask.array<chunksize=(168, 1, 75, 85), meta=np.ndarray>
    salt        (ocean_time, s_rho, eta_rho, xi_rho) float32 46GB dask.array<chunksize=(168, 1, 75, 85), meta=np.ndarray>
    Hwave       (ocean_time, eta_rho, xi_rho) float32 3GB dask.array<chunksize=(168, 75, 85), meta=np.ndarray>
Attributes: (12/37)
    CPP_options:       COAWST, ANA_BPFLUX, ANA_BSFLUX, ANA_BTFLUX, ANA_FSOBC,...
    Conventions:       CF-1.4, SGRID-0.3
    NLM_LBC:           \nEDGE:         WEST   SOUTH  EAST   NORTH  \nzeta:   ...
    ana_file:          ROMS/Functionals/ana_btflux.h, ROMS/Functionals/ana_fs...
    bry_file_01:       ./forcings3/USE_coawst_bdy.nc
    clm_file_01:       ./forcings3/USE_coawst_clm.nc
    ...                ...
    svn_url:            
    tide_file:         ../forcings/tide_forc_USeast_grd16_osu_rev2.nc
    tiling:            006x004
    title:             COAWST ROMS SWAN
    type:              ROMS/TOMS history file
    var_info:          ROMS/External/varinfo.dat

In [32]:
ds_drb_timeslice = ds_drb.cf.sel(T=slice('2022-04-01 00:00','2022-04-08 00:00'))

In [33]:
ds_drb_timeslice = ds_drb_timeslice.chunk({'eta_rho':-1, 'xi_rho':-1})  # chunk to full spatial subset domain
print(f'Uncompressed dataset size: {ds_drb_timeslice.nbytes/1e6} MB')

Uncompressed dataset size: 142.31698 MB


In [34]:
%%time
var = 'salt'
da_drb = ds_drb_timeslice[var].load()

CPU times: user 1.57 s, sys: 763 ms, total: 2.33 s
Wall time: 16.5 s


In [35]:
viz = da_drb.hvplot.quadmesh(x=x.name, y=y.name, geo=True,
                    cmap='turbo', rasterize=True, tiles='OSM', title=var)
viz = pn.panel(viz, widgets={'ocean_time': pn.widgets.Select} )
pn.Column(viz).servable('DRB Explorer')

Column
    [0] Row
        [0] HoloViews(DynamicMap, sizing_mode='fixed', widgets={'ocean_time': <...})
        [1] WidgetBox(align=('end', 'start'))
            [0] DiscreteSlider(name='S-coordinate a..., options={'-0.96875': np.float64(-0...}, value=np.float64(-0.96875))
            [1] Select(name='ocean_time', options={'2022-04-01 00:00:00': np...}, value=np.datetime64('2022-04-01T...)

Close the Dask client since we can't write NetCDF in parallel

In [36]:
client.close()

Specify the encoding to enable compression in the NetCDF file

In [37]:
%%time
encoding={}
for var in ds_drb_timeslice.variables:
    encoding[var] = dict(zlib=True, complevel=4, 
                         fletcher32=False, shuffle=True,
                         _FillValue=None)

ds_drb_timeslice.to_netcdf('drb.nc', encoding=encoding, mode='w')

CPU times: user 17.4 s, sys: 12.1 s, total: 29.5 s
Wall time: 21.7 s


## Stop cluster

In [38]:
cluster.close() if hasattr(cluster, "close") else cluster.shutdown()

2026-05-03 12:34:24,171 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/home/rsignell/miniforge3/envs/coawst-icechunk/lib/python3.12/site-packages/distributed/comm/tcp.py", line 226, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/rsignell/miniforge3/envs/coawst-icechunk/lib/python3.12/site-packages/distributed/worker.py", line 1273, in heartbeat
    response = await retry_operation(
               ^^^^^^^^^^^^^^^^^^^^^^
  File "/home/rsignell/miniforge3/envs/coawst-icechunk/lib/python3.12/site-packages/distributed/utils_comm.py", line 416, in retry_operation
    return await retry(
           ^^^^^^^^^^^^
  File "/home/rsignell/miniforge3/envs/